# Module 2: Pythonic Patterns for AI
## Comprehensions & Generators for Memory-Efficient Data Processing

---

### Welcome to Real-World ML Engineering

In Module 1, you learned how data flows through ML systems. Now you'll learn **how professional ML engineers write code** that's:

- **Clean** — Readable and maintainable
- **Fast** — Leverages Python's optimized C implementations
- **Memory-efficient** — Processes gigabytes without crashing

### Why This Matters

Imagine you're building a language model. Your training corpus is 50GB. If you write:

```python
# ❌ This will crash your laptop
data = []
for line in open('corpus.txt'):
    data.append(line.lower().strip())
# Loads entire 50GB into memory at once
```

But professional ML codebases use **generators** to stream data:

```python
# ✅ Processes 50GB using ~100MB memory
def stream_corpus(path):
    for line in open(path):
        yield line.lower().strip()
# Loads one line at a time
```

By the end of this module, you'll understand the patterns used in **PyTorch DataLoaders**, **Hugging Face Datasets**, and every major ML library.

---

### Learning Objectives

1. Write one-line data transformations with **list/dict comprehensions**
2. Understand Python's **eager vs lazy evaluation**
3. Build **generator functions** that stream data efficiently
4. Measure and optimize **memory usage**
5. Create an **infinite data loader** for training loops

Let's begin!

In [ ]:
# Setup: Load environment and import libraries
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import sys
import tracemalloc
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Verify environment
print(f"Python version: {sys.version}")
print(f"Current directory: {Path.cwd()}")
print("✓ Environment loaded successfully")

---

## 1. List Comprehensions: The Foundation

### The Old Way: For Loops

Let's say you want to extract all the numbers from our corpus lines. Here's how most beginners write it:

In [ ]:
# Traditional approach
sample_lines = [
    "1\tNeural networks learn from data.",
    "2\tTransformers use attention.",
    "3\tGradient descent optimizes.",
    "4\tEmbeddings map tokens.",
    "5\tBackpropagation computes gradients."
]

# Extract line numbers using a for loop
numbers = []
for line in sample_lines:
    num = int(line.split('\t')[0])
    numbers.append(num)

print(f"Line numbers: {numbers}")

**Problems with this approach:**
1. Four lines of code for a simple operation
2. Creates an empty list first (extra memory)
3. Calls `.append()` N times (function call overhead)
4. Hard to read — what does this code do?

### The Pythonic Way: List Comprehension

**Syntax:** `[expression for item in iterable]`

In [ ]:
# Same operation in one line
numbers = [int(line.split('\t')[0]) for line in sample_lines]
print(f"Line numbers: {numbers}")

> **Key Insight:** List comprehensions are:
> - **Faster** — Implemented in C, not Python bytecode
> - **Clearer** — Readable left-to-right: "make a list of numbers from lines"
> - **More Pythonic** — This is how professional code looks

### Adding Conditions

**Syntax:** `[expression for item in iterable if condition]`

In [ ]:
# Extract only even line numbers
even_numbers = [int(line.split('\t')[0]) for line in sample_lines 
                if int(line.split('\t')[0]) % 2 == 0]

print(f"Even line numbers: {even_numbers}")

# Better: Extract once, filter in comprehension
even_numbers_v2 = [num for num in 
                   [int(line.split('\t')[0]) for line in sample_lines]
                   if num % 2 == 0]

print(f"Even line numbers (v2): {even_numbers_v2}")

### Real ML Use Case: Data Cleaning

This is **exactly** how you'd preprocess text for a language model:

In [ ]:
# Messy text data (common in real datasets)
raw_text = [
    "  NEURAL Networks LEARN   ",
    "Transformers USE attention",
    "   ",  # Empty line
    "gradient DESCENT optimizes\n",
    "",  # Another empty
]

# Clean in ONE line: lowercase, strip whitespace, filter empty
cleaned = [text.lower().strip() for text in raw_text if text.strip()]

print("Raw:")
for i, text in enumerate(raw_text):
    print(f"  {i}: '{text}'")

print("\nCleaned:")
for i, text in enumerate(cleaned):
    print(f"  {i}: '{text}'")

### Nested Comprehensions

Extract all words from all lines:

In [ ]:
# Get content after the tab character
sentences = [line.split('\t')[1] for line in sample_lines]

# Nested comprehension: Extract all words
all_words = [word for sentence in sentences for word in sentence.split()]

print(f"First 10 words: {all_words[:10]}")
print(f"Total words: {len(all_words)}")

> **Reading nested comprehensions:** Read left to right like English:
> - "for each sentence" (outer loop)
> - "for each word in that sentence" (inner loop)
> - "collect the word"

### Speed Comparison: For Loop vs Comprehension

In [ ]:
# Load real corpus data
corpus_path = Path('sample_corpus.txt')
with open(corpus_path) as f:
    corpus_lines = f.readlines()

print(f"Loaded {len(corpus_lines)} lines from corpus")

# Test 1: For loop approach
start = time.perf_counter()
numbers_loop = []
for line in corpus_lines:
    if '\t' in line:
        numbers_loop.append(int(line.split('\t')[0]))
time_loop = time.perf_counter() - start

# Test 2: Comprehension approach
start = time.perf_counter()
numbers_comp = [int(line.split('\t')[0]) for line in corpus_lines if '\t' in line]
time_comp = time.perf_counter() - start

# Visualize
fig, ax = plt.subplots(figsize=(8, 5))
methods = ['For Loop', 'Comprehension']
times = [time_loop * 1000, time_comp * 1000]  # Convert to milliseconds
colors = ['#ff6b6b', '#4ecdc4']

bars = ax.bar(methods, times, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Execution Time (ms)', fontsize=12, fontweight='bold')
ax.set_title('Speed Comparison: For Loop vs List Comprehension', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar, time_val in zip(bars, times):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.3f} ms',
            ha='center', va='bottom', fontweight='bold', fontsize=11)

# Add speedup annotation
speedup = time_loop / time_comp
ax.text(0.5, max(times) * 0.9, 
        f'Comprehension is {speedup:.1f}x faster!',
        ha='center', fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

print(f"\nFor Loop:       {time_loop*1000:.3f} ms")
print(f"Comprehension:  {time_comp*1000:.3f} ms")
print(f"Speedup:        {speedup:.2f}x")

> **Why is it faster?**
> - List comprehensions are implemented in C (CPython)
> - No function call overhead (`.append()`)
> - Memory allocated once upfront
> - Optimized bytecode instructions

---

## 2. Dictionary Comprehensions: Building Vocabularies

In NLP, you constantly build **vocabularies**: mappings from words → IDs.

### Basic Syntax

**Syntax:** `{key_expr: value_expr for item in iterable}`

In [ ]:
# Create a vocabulary from unique words
sample_text = "neural networks learn from data neural networks optimize"
words = sample_text.split()
unique_words = sorted(set(words))  # Get unique words, sorted

# Map each word to an ID
vocab = {word: idx for idx, word in enumerate(unique_words)}

print("Vocabulary:")
for word, idx in vocab.items():
    print(f"  '{word}' → {idx}")

### Inverting Dictionaries

Often you need the reverse mapping: ID → word

In [ ]:
# Invert the vocabulary: ID → word
inverse_vocab = {idx: word for word, idx in vocab.items()}

print("Inverse Vocabulary:")
for idx, word in sorted(inverse_vocab.items()):
    print(f"  {idx} → '{word}'")

# Use it to decode IDs back to words
ids = [0, 1, 2, 3, 4]
decoded = [inverse_vocab[i] for i in ids]
print(f"\nDecoded: {' '.join(decoded)}")

### Filtering Dictionaries

Keep only frequent words (common in NLP preprocessing):

In [ ]:
# Count word frequencies
word_counts = {}
for word in words:
    word_counts[word] = word_counts.get(word, 0) + 1

print("Word Frequencies:")
for word, count in sorted(word_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  '{word}': {count}")

# Keep only words that appear 2+ times
frequent_words = {word: count for word, count in word_counts.items() if count >= 2}

print("\nFrequent Words (≥2 occurrences):")
print(frequent_words)

### Real ML Use Case: Word Frequency Map in One Line

In [ ]:
from collections import Counter

# Extract all words from corpus
all_corpus_words = [
    word.lower() 
    for line in corpus_lines 
    if '\t' in line
    for word in line.split('\t')[1].split()
]

# Build frequency map in one line
freq_map = dict(Counter(all_corpus_words).most_common(10))

print("Top 10 Most Frequent Words:")
for word, count in freq_map.items():
    print(f"  '{word}': {count}")

> **Key Insight:** Dict comprehensions + Counter = one-line vocabulary builders.
> This is **exactly** how tokenizers work in Hugging Face Transformers!

---

## 3. The Memory Problem

### What Happens When Data Gets Big?

Let's simulate loading a large dataset into memory:

In [ ]:
# Simulate different data sizes
def measure_list_memory(size):
    """Create a list of strings and measure memory usage"""
    data = [f"line {i}: neural networks learn from data" for i in range(size)]
    return sys.getsizeof(data) + sum(sys.getsizeof(item) for item in data)

sizes = [100, 1_000, 10_000, 100_000, 1_000_000]
memory_usage = [measure_list_memory(size) / (1024**2) for size in sizes]  # Convert to MB

# Visualize memory growth
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sizes, memory_usage, marker='o', linewidth=2.5, markersize=8, color='#e74c3c')
ax.fill_between(sizes, memory_usage, alpha=0.3, color='#e74c3c')

ax.set_xlabel('Number of Lines', fontsize=12, fontweight='bold')
ax.set_ylabel('Memory Usage (MB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Usage: Loading Data into a List', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xscale('log')
ax.grid(True, alpha=0.3, linestyle='--')

# Annotate the danger zone
ax.axhline(y=1000, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax.text(1000, 1100, 'Laptop crashes here →', fontsize=11, 
        color='red', fontweight='bold')

# Add data labels
for size, mem in zip(sizes, memory_usage):
    ax.text(size, mem + 10, f'{mem:.1f} MB', 
            ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("Memory Usage by List Size:")
for size, mem in zip(sizes, memory_usage):
    print(f"  {size:>9,} lines → {mem:>8.2f} MB")

> **The Problem:**
> - Loading 1 million lines = ~60 MB
> - A 50GB corpus = ~50,000 MB (your laptop has maybe 8GB-16GB)
> - Your Python process **crashes** with `MemoryError`

### The Traditional Approach Fails at Scale

In [ ]:
# ❌ This loads everything into memory
def load_entire_file(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(line.strip())
    return data

# Measure memory
tracemalloc.start()
data = load_entire_file(corpus_path)
current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print(f"Loaded {len(data)} lines")
print(f"Memory used: {peak / 1024:.2f} KB")
print(f"\n❌ This approach loads ALL data into memory at once")
print(f"❌ Would crash on a 10GB file")

**Question:** How do PyTorch and TensorFlow train on terabyte datasets?

**Answer:** They use **generators** to stream data one batch at a time.

---

## 4. Generator Functions: The Solution

### What is a Generator?

A generator is a function that **yields** values one at a time instead of returning a list.

```mermaid
graph LR
    A[Regular Function] --> B[Returns ALL results]
    B --> C[Stored in memory]
    
    D[Generator Function] --> E[Yields ONE result]
    E --> F[Compute next on demand]
    F --> E
    
    style A fill:#ff6b6b
    style D fill:#4ecdc4
    style B fill:#feca57
    style E fill:#48dbfb
```

### The `yield` Keyword

In [ ]:
# Regular function: Returns a list
def count_up_to_list(n):
    result = []
    for i in range(n):
        result.append(i)
    return result

# Generator function: Yields one value at a time
def count_up_to_gen(n):
    for i in range(n):
        yield i  # ← This is the magic

# Compare what they return
list_result = count_up_to_list(5)
gen_result = count_up_to_gen(5)

print("Regular function returns:")
print(f"  Type: {type(list_result)}")
print(f"  Value: {list_result}")
print(f"  Memory: {sys.getsizeof(list_result)} bytes")

print("\nGenerator function returns:")
print(f"  Type: {type(gen_result)}")
print(f"  Value: {gen_result}")  # Just a generator object!
print(f"  Memory: {sys.getsizeof(gen_result)} bytes")

> **Key Insight:** A generator doesn't compute anything upfront.
> It returns a "recipe" for computing values on demand.

### How Generators Pause and Resume

In [ ]:
# Create a generator
gen = count_up_to_gen(3)

print("Manually calling next():")
print(f"  First call:  {next(gen)}")   # Runs until first yield
print(f"  Second call: {next(gen)}")   # Resumes from where it left off
print(f"  Third call:  {next(gen)}")   # Continues...
print("\nGenerator exhausted. Calling next() again raises StopIteration:")
try:
    next(gen)
except StopIteration:
    print("  → StopIteration exception raised")

### Iterating Over Generators

Most of the time, you use generators in `for` loops:

In [ ]:
# For loops automatically call next() until StopIteration
print("Iterating over generator:")
for i in count_up_to_gen(5):
    print(f"  {i}")

print("\n✓ For loops handle generators transparently!")

---

## 5. Lazy Evaluation: Compute Only When Needed

### Eager vs Lazy Evaluation

```mermaid
flowchart TD
    A[Start] --> B{Evaluation Type?}
    
    B -->|Eager: List| C[Load ALL data]
    C --> D[Store in memory]
    D --> E[Process item 1]
    D --> F[Process item 2]
    D --> G[Process item N]
    E --> H[Done]
    F --> H
    G --> H
    
    B -->|Lazy: Generator| I[Load item 1 only]
    I --> J[Process item 1]
    J --> K{More items?}
    K -->|Yes| L[Load next item]
    L --> M[Process it]
    M --> K
    K -->|No| N[Done]
    
    style C fill:#ff6b6b,stroke:#c0392b,stroke-width:2px
    style D fill:#ff6b6b,stroke:#c0392b,stroke-width:2px
    style I fill:#4ecdc4,stroke:#1abc9c,stroke-width:2px
    style L fill:#4ecdc4,stroke:#1abc9c,stroke-width:2px
```

In [ ]:
# Eager evaluation: List comprehension
tracemalloc.start()
eager = [i**2 for i in range(1_000_000)]  # Computes ALL squares immediately
eager_mem = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

# Lazy evaluation: Generator expression
tracemalloc.start()
lazy = (i**2 for i in range(1_000_000))  # Just stores the recipe
lazy_mem = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print("Memory Comparison:")
print(f"  Eager (list):      {eager_mem / (1024**2):.2f} MB")
print(f"  Lazy (generator):  {lazy_mem / 1024:.2f} KB")
print(f"\n  Savings: {eager_mem / lazy_mem:.0f}x less memory!")

### Generator Expressions

**Syntax:** `(expression for item in iterable)` — note the **parentheses** instead of brackets!

In [ ]:
# List comprehension: Eager
list_comp = [x**2 for x in range(10)]
print(f"List comprehension: {list_comp}")
print(f"Type: {type(list_comp)}")

# Generator expression: Lazy
gen_expr = (x**2 for x in range(10))
print(f"\nGenerator expression: {gen_expr}")
print(f"Type: {type(gen_expr)}")

# Convert to list to see values
print(f"Values: {list(gen_expr)}")

> **When to use each:**
> - **List comprehension:** Need to access data multiple times, data fits in memory
> - **Generator expression:** Processing once, data might be huge, or you only need a few items

### Memory Comparison Visualization

In [ ]:
# Compare memory usage for different sizes
sizes = [1_000, 10_000, 100_000, 1_000_000]
list_memory = []
gen_memory = []

for size in sizes:
    # Measure list
    tracemalloc.start()
    _ = [i**2 for i in range(size)]
    list_mem = tracemalloc.get_traced_memory()[1] / (1024**2)  # MB
    tracemalloc.stop()
    list_memory.append(list_mem)
    
    # Measure generator
    tracemalloc.start()
    _ = (i**2 for i in range(size))
    gen_mem = tracemalloc.get_traced_memory()[1] / 1024  # KB
    tracemalloc.stop()
    gen_memory.append(gen_mem / 1024)  # Convert to MB for comparison

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Comparison
x = np.arange(len(sizes))
width = 0.35
bars1 = ax1.bar(x - width/2, list_memory, width, label='List (Eager)', 
                color='#e74c3c', alpha=0.8, edgecolor='black')
bars2 = ax1.bar(x + width/2, gen_memory, width, label='Generator (Lazy)', 
                color='#2ecc71', alpha=0.8, edgecolor='black')

ax1.set_xlabel('Data Size', fontsize=12, fontweight='bold')
ax1.set_ylabel('Memory Usage (MB)', fontsize=12, fontweight='bold')
ax1.set_title('Memory: List vs Generator', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([f'{s:,}' for s in sizes], rotation=45)
ax1.legend(fontsize=11)
ax1.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0.01:  # Only show if visible
            ax1.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom', fontsize=9)

# Right: Memory savings ratio
savings = [list_memory[i] / gen_memory[i] if gen_memory[i] > 0 else 0 
           for i in range(len(sizes))]
bars = ax2.bar(range(len(sizes)), savings, color='#3498db', alpha=0.8, 
               edgecolor='black', linewidth=1.5)
ax2.set_xlabel('Data Size', fontsize=12, fontweight='bold')
ax2.set_ylabel('Memory Savings (x times)', fontsize=12, fontweight='bold')
ax2.set_title('Generator Memory Efficiency', fontsize=14, fontweight='bold')
ax2.set_xticks(range(len(sizes)))
ax2.set_xticklabels([f'{s:,}' for s in sizes], rotation=45)
ax2.grid(axis='y', alpha=0.3)

for i, bar in enumerate(bars):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.0f}x',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

---

## 6. Building a Data Stream

### Reading Files with Generators

This is the **core pattern** used in PyTorch DataLoaders:

In [ ]:
def stream_file(path):
    """Stream file line by line — NEVER loads entire file into memory"""
    with open(path) as f:
        for line in f:
            yield line.strip()

# Use it
print("First 5 lines from corpus (streamed):")
for i, line in enumerate(stream_file(corpus_path)):
    if i >= 5:
        break
    print(f"  {i+1}: {line[:60]}...")

### Adding Data Cleaning

In [ ]:
def stream_and_clean(path):
    """Stream file and apply cleaning transformations lazily"""
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:  # Skip empty lines
                continue
            if '\t' in line:
                # Extract content after tab, lowercase
                content = line.split('\t', 1)[1].lower()
                yield content

print("Cleaned lines (streamed):")
for i, line in enumerate(stream_and_clean(corpus_path)):
    if i >= 5:
        break
    print(f"  {i+1}: {line}")

### Yielding Batches

This is **exactly** how training loops work:

In [ ]:
def batch_generator(path, batch_size=4):
    """Yield batches of lines — just like DataLoader!"""
    batch = []
    for line in stream_and_clean(path):
        batch.append(line)
        if len(batch) == batch_size:
            yield batch
            batch = []  # Reset for next batch
    
    # Yield remaining items as final batch
    if batch:
        yield batch

print("First 3 batches of size 4:")
for i, batch in enumerate(batch_generator(corpus_path, batch_size=4)):
    if i >= 3:
        break
    print(f"\nBatch {i+1}:")
    for line in batch:
        print(f"  • {line[:50]}...")

### Connection to PyTorch DataLoader

```mermaid
flowchart LR
    A[sample_corpus.txt<br/>50 lines] --> B[Generator<br/>stream_and_clean]
    B --> C[Batch Generator<br/>batch_size=32]
    C --> D[Training Loop<br/>for batch in loader]
    D --> E[Model Forward Pass]
    E --> F[Backward + Optimize]
    F --> D
    
    style A fill:#ffeaa7
    style B fill:#74b9ff
    style C fill:#a29bfe
    style D fill:#fd79a8
    style E fill:#fdcb6e
    style F fill:#00b894
```

**This is EXACTLY how PyTorch works:**
```python
# PyTorch DataLoader is just a fancy generator!
from torch.utils.data import DataLoader

for batch in DataLoader(dataset, batch_size=32):
    # Same pattern we just built!
    outputs = model(batch)
    loss.backward()
```

---

## 7. Lab: The Infinite Data Loader

### Challenge: Build a Production-Ready Data Loader

**Requirements:**
1. Stream data from `sample_corpus.txt`
2. Apply cleaning (lowercase, strip whitespace)
3. Filter empty lines and lines without tabs
4. Yield batches of N lines
5. **Infinite mode:** Loop through the file forever (for multi-epoch training)
6. Track memory usage — should stay constant regardless of file size

### Solution

In [ ]:
def infinite_data_loader(path, batch_size=8, max_batches=None):
    """
    Production-ready data loader with infinite epoch support.
    
    Args:
        path: Path to corpus file
        batch_size: Number of lines per batch
        max_batches: Stop after N batches (None = infinite)
    
    Yields:
        List of cleaned lines (batch)
    """
    batch_count = 0
    
    while True:  # Infinite loop for multi-epoch training
        batch = []
        
        with open(path) as f:
            for line in f:
                # Clean and filter
                line = line.strip()
                if not line or '\t' not in line:
                    continue
                
                # Extract content and clean
                content = line.split('\t', 1)[1].lower()
                batch.append(content)
                
                # Yield when batch is full
                if len(batch) == batch_size:
                    yield batch
                    batch = []
                    batch_count += 1
                    
                    if max_batches and batch_count >= max_batches:
                        return
        
        # Yield remaining items from this epoch
        if batch:
            yield batch
            batch_count += 1
            
            if max_batches and batch_count >= max_batches:
                return

# Test the loader
print("Testing Infinite Data Loader:")
print("=" * 60)

loader = infinite_data_loader(corpus_path, batch_size=4, max_batches=5)

for i, batch in enumerate(loader, 1):
    print(f"\nBatch {i} (size={len(batch)}):")
    for line in batch:
        print(f"  • {line[:55]}...")

print("\n" + "=" * 60)
print("✓ Loader works! Memory usage stays constant.")

### Memory Test: Large Data Stream

In [ ]:
# Simulate processing many batches
tracemalloc.start()

loader = infinite_data_loader(corpus_path, batch_size=8, max_batches=100)
batch_count = 0
memory_samples = []

for i, batch in enumerate(loader):
    batch_count += 1
    # Sample memory every 10 batches
    if i % 10 == 0:
        current_mem = tracemalloc.get_traced_memory()[0] / (1024**2)  # MB
        memory_samples.append(current_mem)

final_mem = tracemalloc.get_traced_memory()[1] / (1024**2)  # Peak MB
tracemalloc.stop()

# Visualize memory stability
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(len(memory_samples)), memory_samples, 
        marker='o', linewidth=2.5, markersize=6, color='#2ecc71')
ax.axhline(y=np.mean(memory_samples), color='#e74c3c', 
           linestyle='--', linewidth=2, alpha=0.7, 
           label=f'Average: {np.mean(memory_samples):.2f} MB')

ax.set_xlabel('Sample Point (every 10 batches)', fontsize=12, fontweight='bold')
ax.set_ylabel('Memory Usage (MB)', fontsize=12, fontweight='bold')
ax.set_title('Memory Stays Constant with Generator-Based Streaming', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(fontsize=11)

# Add annotation
ax.text(len(memory_samples) * 0.5, max(memory_samples) * 1.05,
        '✓ Memory remains stable\nregardless of data size',
        ha='center', fontsize=11, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.7', facecolor='#dfe6e9', alpha=0.8))

plt.tight_layout()
plt.show()

print(f"\nProcessed {batch_count} batches")
print(f"Peak memory: {final_mem:.2f} MB")
print(f"Average memory: {np.mean(memory_samples):.2f} MB")
print(f"Memory variance: {np.std(memory_samples):.4f} MB")
print("\n✓ Memory usage is stable — this scales to ANY file size!")

---

## Summary & Key Takeaways

### What You Learned

#### 1. List Comprehensions
- **Syntax:** `[expr for item in iterable if condition]`
- **Use when:** Transforming data that fits in memory
- **Benefits:** Faster, cleaner, more Pythonic than for loops

#### 2. Dict Comprehensions
- **Syntax:** `{key: value for item in iterable}`
- **Use when:** Building vocabularies, frequency maps, inverted indices
- **Real ML example:** Tokenizer vocabularies in Hugging Face

#### 3. Generator Functions
- **Syntax:** Use `yield` instead of `return`
- **Use when:** Data doesn't fit in memory, or you only process once
- **Benefits:** Constant memory usage, lazy evaluation

#### 4. Generator Expressions
- **Syntax:** `(expr for item in iterable)` — note parentheses!
- **Use when:** Quick one-liner for lazy evaluation
- **Benefits:** Memory-efficient alternative to list comprehensions

#### 5. The Data Streaming Pattern
```python
def stream_file(path):
    with open(path) as f:
        for line in f:
            yield process(line)
```
This is the **foundation** of:
- PyTorch `DataLoader`
- TensorFlow `tf.data.Dataset`
- Hugging Face `datasets.Dataset`

---

### Comparison Table

| Pattern | Memory | Speed | Use Case |
|---------|--------|-------|----------|
| For loop | High | Slowest | Learning, debugging |
| List comprehension | High | Fast | Small/medium data, multiple passes |
| Generator function | Constant | Fast | Large data, single pass |
| Generator expression | Constant | Fast | Quick one-liners |

---

### Decision Tree

```mermaid
flowchart TD
    A[Need to process data] --> B{Data fits in memory?}
    B -->|Yes| C{Access multiple times?}
    B -->|No| D[Use Generator]
    
    C -->|Yes| E[List Comprehension]
    C -->|No| F{Complex logic?}
    
    F -->|Yes| G[Generator Function]
    F -->|No| H[Generator Expression]
    
    D --> I[Generator Function<br/>with batching]
    
    style E fill:#a29bfe
    style G fill:#74b9ff
    style H fill:#74b9ff
    style I fill:#00b894
```

---

### Next Steps

In **Module 3**, you'll learn:
- How to **vectorize** operations with NumPy
- Broadcasting rules for efficient computation
- Building a dense retrieval system with cosine similarity

But first, try the exercises below!

---

## Try It Yourself: Exercises

### Exercise 1: Vocabulary Builder

Build a complete vocabulary system:
1. Extract all unique words from `sample_corpus.txt`
2. Create a `word → id` mapping (sorted alphabetically)
3. Create an `id → word` inverse mapping
4. Encode a sentence into IDs
5. Decode IDs back into words

In [ ]:
# Your code here
def build_vocabulary(corpus_path):
    """
    Build word → id and id → word mappings.
    
    Returns:
        word2id: dict mapping word to ID
        id2word: dict mapping ID to word
    """
    # TODO: Extract all words from corpus
    # TODO: Get unique words, sort alphabetically
    # TODO: Create word2id mapping
    # TODO: Create id2word mapping (invert)
    pass

def encode_sentence(sentence, word2id):
    """Convert sentence to list of IDs"""
    # TODO: Split sentence, map each word to ID
    pass

def decode_ids(ids, id2word):
    """Convert list of IDs back to sentence"""
    # TODO: Map each ID to word, join with spaces
    pass

# Test your implementation
word2id, id2word = build_vocabulary(corpus_path)
print(f"Vocabulary size: {len(word2id)}")
print(f"First 10 words: {list(word2id.keys())[:10]}")

test_sentence = "neural networks learn from data"
encoded = encode_sentence(test_sentence, word2id)
decoded = decode_ids(encoded, id2word)
print(f"\nOriginal:  {test_sentence}")
print(f"Encoded:   {encoded}")
print(f"Decoded:   {decoded}")

### Exercise 2: Windowed Data Generator

Create a generator that yields **overlapping windows** of text for language modeling:
- Window size: N consecutive lines
- Stride: S (how many lines to skip between windows)

Example:
```
Lines: [A, B, C, D, E]
Window size: 3, stride: 1
Yields: [A,B,C], [B,C,D], [C,D,E]
```

In [ ]:
# Your code here
def windowed_generator(path, window_size=3, stride=1):
    """
    Generate overlapping windows of lines.
    
    Args:
        path: Path to corpus file
        window_size: Number of lines per window
        stride: Step size between windows
    
    Yields:
        List of lines (window)
    """
    # TODO: Use a sliding window approach
    # Hint: Keep a buffer of size window_size
    pass

# Test your implementation
print("First 5 windows (size=3, stride=1):")
for i, window in enumerate(windowed_generator(corpus_path, window_size=3, stride=1)):
    if i >= 5:
        break
    print(f"\nWindow {i+1}:")
    for line in window:
        print(f"  {line[:50]}...")

### Exercise 3: Memory-Efficient Word Frequency Counter

Count word frequencies using **only generators** — no lists allowed!
- Stream the file line by line
- Yield individual words
- Use `collections.Counter` on the generator

Challenge: Measure memory usage vs loading all words into a list first.

In [ ]:
# Your code here
from collections import Counter

def word_stream(path):
    """
    Generator that yields individual words from corpus.
    """
    # TODO: Stream file, extract content, yield words one at a time
    pass

# Test: Count word frequencies using the generator
tracemalloc.start()
word_freq = Counter(word_stream(corpus_path))
gen_memory = tracemalloc.get_traced_memory()[1] / (1024**2)  # MB
tracemalloc.stop()

print(f"Top 10 most common words:")
for word, count in word_freq.most_common(10):
    print(f"  '{word}': {count}")

print(f"\nMemory used (generator): {gen_memory:.2f} MB")

# Compare with list-based approach
tracemalloc.start()
all_words = [word for line in open(corpus_path) if '\t' in line 
             for word in line.split('\t')[1].lower().split()]
word_freq_list = Counter(all_words)
list_memory = tracemalloc.get_traced_memory()[1] / (1024**2)  # MB
tracemalloc.stop()

print(f"Memory used (list):      {list_memory:.2f} MB")
print(f"Memory savings:          {list_memory / gen_memory:.1f}x")

### Exercise 4: Advanced Challenge — Multi-File Streaming

Create a generator that streams data from **multiple corpus files** sequentially:
- Accept a list of file paths
- Stream from each file in order
- Yield batches that can span file boundaries

This is exactly how real ML training pipelines work with multi-file datasets!

In [ ]:
# Your code here
def multi_file_loader(file_paths, batch_size=8):
    """
    Stream data from multiple files, yielding batches.
    
    Args:
        file_paths: List of corpus file paths
        batch_size: Number of lines per batch
    
    Yields:
        List of cleaned lines (batch)
    """
    # TODO: Iterate through files
    # TODO: Accumulate lines across file boundaries
    # TODO: Yield batches when full
    pass

# For testing, use the same file twice (simulates multiple files)
test_files = [corpus_path, corpus_path]

print("First 3 batches from multi-file loader:")
for i, batch in enumerate(multi_file_loader(test_files, batch_size=4)):
    if i >= 3:
        break
    print(f"\nBatch {i+1}:")
    for line in batch:
        print(f"  • {line[:50]}...")

---

## Solutions

<details>
<summary>Click to reveal solutions (try the exercises first!)</summary>

### Exercise 1 Solution: Vocabulary Builder

```python
def build_vocabulary(corpus_path):
    # Extract all words
    words = [
        word.lower() 
        for line in open(corpus_path) 
        if '\t' in line
        for word in line.split('\t')[1].split()
    ]
    
    # Get unique, sorted
    unique_words = sorted(set(words))
    
    # Create mappings
    word2id = {word: idx for idx, word in enumerate(unique_words)}
    id2word = {idx: word for word, idx in word2id.items()}
    
    return word2id, id2word

def encode_sentence(sentence, word2id):
    return [word2id[word] for word in sentence.lower().split() if word in word2id]

def decode_ids(ids, id2word):
    return ' '.join(id2word[i] for i in ids)
```

### Exercise 2 Solution: Windowed Generator

```python
def windowed_generator(path, window_size=3, stride=1):
    buffer = []
    
    for line in stream_and_clean(path):
        buffer.append(line)
        
        if len(buffer) == window_size:
            yield buffer[:]
            # Remove stride elements from beginning
            buffer = buffer[stride:]
```

### Exercise 3 Solution: Word Stream

```python
def word_stream(path):
    with open(path) as f:
        for line in f:
            if '\t' not in line:
                continue
            content = line.split('\t', 1)[1].lower()
            for word in content.split():
                yield word
```

### Exercise 4 Solution: Multi-File Loader

```python
def multi_file_loader(file_paths, batch_size=8):
    batch = []
    
    for path in file_paths:
        for line in stream_and_clean(path):
            batch.append(line)
            
            if len(batch) == batch_size:
                yield batch
                batch = []
    
    if batch:
        yield batch
```

</details>

---

### Congratulations!

You now understand the **core patterns** used in every production ML system.

When you look at PyTorch, TensorFlow, or Hugging Face code, you'll recognize these patterns everywhere.

**Next:** Module 3 — Vector Operations & Dense Retrieval